In [1]:
!pip install datasets PySastrawi pandas


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from datasets import load_dataset

# 1. Memuat dataset dari Hugging Face
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")
df = pd.DataFrame(dataset['train'])

# 2. Menampilkan jumlah data, daftar nama kolom, dan 5 data teratas
print("Jumlah total data:", len(df))
print("Daftar nama kolom:", list(df.columns))
display(df.head())

# 3. Menampilkan distribusi label sentimen dari kolom 'sentimen'
print("\nDistribusi Label Sentimen:")
print(df['sentimen'].value_counts())

C:\Users\ASUS TUF GAMING\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Jumlah total data: 11606
Daftar nama kolom: ['review', 'translate', 'sentimen']


,review,translate,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,sound and mic satisfied. I only ordered the on...,0
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay a...",0
3,Kualitas produk baik. Harga terjangkau dan pen...,product quality is good. affordable prices and...,0
4,"Overall bagus, tombol berfungsi semuanya, tapi...","overall is good, the buttons all work, but the...",0



Distribusi Label Sentimen:
sentimen
0    5803
1    5803
Name: count, dtype: int64


In [3]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi daftar stopword dan stemmer Bahasa Indonesia
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocessing_teks(teks):
    # 1. Case folding: mengubah teks menjadi huruf kecil
    teks = str(teks).lower()
    
    # 2. Cleaning: menghapus URL, angka, tanda baca, dan karakter khusus
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks)
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    
    # 3. Tokenisasi: memisahkan teks menjadi kata
    tokens = teks.split()
    
    # 4. Stopword removal: menghapus kata yang tidak penting
    tokens = [kata for kata in tokens if kata not in stopwords]
    
    # 5. Stemming: mengubah kata menjadi bentuk dasar
    hasil_stem = [stemmer.stem(kata) for kata in tokens]
    
    return " ".join(hasil_stem)

print("Fungsi preprocessing siap digunakan!")

Fungsi preprocessing siap digunakan!


In [4]:
# Mengatur tampilan kolom agar teks tidak terpotong tanda '...' saat di-screenshot
pd.set_option('display.max_colwidth', None)

# Menerapkan preprocessing pada kolom 'review'
df['review_bersih'] = df['review'].apply(preprocessing_teks)

# Menampilkan 10 contoh teks sebelum dan sesudah preprocessing
tabel_hasil = df[['review', 'review_bersih']].head(10)
display(tabel_hasil)

,review,review_bersih
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg ga nyala
1,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek tolong perhati
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk harga jangkau kirim cepet banget udh order oke tp sayang klo kado gabisa nitip pake kartu ucap bungkus pake kertas kado
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol fungsi packaging penyot robek
5,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk original kualitas cepat kirim sanagt cepat recommended seller
6,kualitas sangat jelek. bocor,kualitas jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih ka barang terima sukses ka
8,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller",body berat cm spiral tipis banding cm moga awet beli cm penasaran aja cm udah rusak mesin harga murah cm fast delivery courier ramah ty seller
9,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesan terima tanggal january buka pagi tanggal january tab sketsa advan nyala cas posisi tab nyala


### Pengaruh Preprocessing terhadap Kualitas Data Teks
1. Reduksi Derau Data (Noise Reduction): Tahap cleaning menghapus unsur yang tidak relevan seperti tanda baca, angka, simbol, dan URL sehingga teks menjadi lebih bersih dan mudah diproses.
2. Penyeragaman Bentuk Kata (Word Normalization): Tahap case folding mengubah seluruh huruf menjadi bentuk yang seragam, sedangkan stemming mengubah kata berimbuhan menjadi kata dasar. Proses ini mengurangi variasi penulisan dan jumlah kosakata unik (vocabulary).
3. Peningkatan Relevansi Informasi (Information Enhancement): Tahap stopword removal menghilangkan kata-kata umum yang tidak memiliki makna signifikan, seperti yang, dan, dan di, sehingga kata-kata penting yang merepresentasikan sentimen atau topik menjadi lebih dominan dalam proses analisis.